## Importing Packages

In [1]:

#initializations
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

## Defining Project Variables

Main reason we are starting at 2007 currently instead of 2000 is because a lot of our ETFs and newer stocks do not have full data back to 2000.

In [2]:
START_DATE = "2007-01-01"
END_DATE = "2026-06-28"
TRADING_DAYS = 252
RISK_FREE_RATE = 0.00

## Define Portfolio Construction


In [3]:
#defining each portfolio 
portfolio_names = {
    "Barbell Portfolio": {
        "Short-Term Treasuries": 0.95,
        "Volatility Proxy": 0.05
    },

    "Mega-Cap Technology Portfolio": {
        "Apple": 0.125,
        "Microsoft": 0.125,
        "Nvidia": 0.125,
        "Amazon": 0.125,
        "Google": 0.125,
        "Meta": 0.125,
        "Broadcom": 0.125,
        "Tesla": 0.125
    },

    "Energy Portfolio": {
        "ExxonMobil": 0.10,
        "Chevron": 0.10,
        "ConocoPhillips": 0.10,
        "EOG Resources": 0.10,
        "Schlumberger": 0.10,
        "NextEra Energy": 0.10,
        "Duke Energy": 0.10,
        "Constellation Energy": 0.10,
        "Southern Company": 0.10,
        "Dominion Energy": 0.10
    },

    "Real Estate Portfolio": {
        "Real Estate ETF": 1.00
    },

    "Consumer Staples Portfolio": {
        "Walmart": 0.10,
        "Costco": 0.10,
        "Target": 0.10,
        "Procter & Gamble": 0.10,
        "Unilever": 0.10,
        "Kimberly-Clark": 0.10,
        "Coca-Cola": 0.10,
        "PepsiCo": 0.10,
        "Nestle": 0.10,
        "General Mills": 0.10
    }
}
# making a ticker look up table as a bridge
ticker_lookup = {
    # Barbell Portfolio
    "Short-Term Treasuries": "TBIL",
    "Volatility Proxy": "^VIX",

    # Mega-Cap Technology Portfolio
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Nvidia": "NVDA",
    "Amazon": "AMZN",
    "Google": "GOOG",
    "Meta": "META",
    "Broadcom": "AVGO",
    "Tesla": "TSLA",

    # Energy Portfolio
    "ExxonMobil": "XOM",
    "Chevron": "CVX",
    "ConocoPhillips": "COP",
    "EOG Resources": "EOG",
    "Schlumberger": "SLB",
    "NextEra Energy": "NEE",
    "Duke Energy": "DUK",
    "Constellation Energy": "CEG",
    "Southern Company": "SO",
    "Dominion Energy": "D",

    # Real Estate Portfolio
    "Real Estate ETF": "VNQ",

    # Consumer Staples Portfolio
    "Walmart": "WMT",
    "Costco": "COST",
    "Target": "TGT",
    "Procter & Gamble": "PG",
    "Unilever": "UL",
    "Kimberly-Clark": "KMB",
    "Coca-Cola": "KO",
    "PepsiCo": "PEP",
    "Nestle": "NSRGY",
    "General Mills": "GIS"
}

# Convert the english portfolios into ticker-based portfolios
portfolios = {}

for portfolio_name, holdings in portfolio_names.items():
    ticker_based_holdings = {}

    for asset_name, weight in holdings.items():
        ticker = ticker_lookup[asset_name]
        ticker_based_holdings[ticker] = weight

    portfolios[portfolio_name] = ticker_based_holdings

# Create readable portfolio reference table
portfolio_reference_rows = []

for portfolio_name, holdings in portfolio_names.items():
    for asset_name, weight in holdings.items():
        portfolio_reference_rows.append({
            "Portfolio": portfolio_name,
            "Asset Name": asset_name,
            "Ticker": ticker_lookup[asset_name],
            "Weight": weight
        })

portfolio_reference = pd.DataFrame(portfolio_reference_rows)

portfolio_reference

,Portfolio,Asset Name,Ticker,Weight
0,Barbell Portfolio,Short-Term Treasuries,TBIL,0.9500
1,Barbell Portfolio,Volatility Proxy,^VIX,0.0500
2,Mega-Cap Technology Portfolio,Apple,AAPL,0.1250
3,Mega-Cap Technology Portfolio,Microsoft,MSFT,0.1250
4,Mega-Cap Technology Portfolio,Nvidia,NVDA,0.1250
5,Mega-Cap Technology Portfolio,Amazon,AMZN,0.1250
6,Mega-Cap Technology Portfolio,Google,GOOG,0.1250
7,Mega-Cap Technology Portfolio,Meta,META,0.1250
8,Mega-Cap Technology Portfolio,Broadcom,AVGO,0.1250
9,Mega-Cap Technology Portfolio,Tesla,TSLA,0.1250


## Benchmark Section

In [4]:
benchmark_names = {
    "S&P 500 ETF": "SPY",
    "Nasdaq 100 ETF": "QQQ",
    "Bond Market ETF": "BND"
}

benchmark_portfolio_names = {
    "60/40 Portfolio": {
        "S&P 500 ETF": 0.60,
        "Bond Market ETF": 0.40
    }
}

# Convert benchmark portfolio names into ticker-based portfolio
benchmark_portfolios = {}

for portfolio_name, holdings in benchmark_portfolio_names.items():
    ticker_based_holdings = {}

    for asset_name, weight in holdings.items():
        ticker = benchmark_names[asset_name]
        ticker_based_holdings[ticker] = weight

    benchmark_portfolios[portfolio_name] = ticker_based_holdings

# Individual benchmark tickers
benchmarks = {
    "S&P 500": "SPY",
    "Nasdaq 100": "QQQ",
    "Bonds": "BND"
}

benchmark_portfolios

{'60/40 Portfolio': {'SPY': 0.6, 'BND': 0.4}}

## Combining the Tickers

In [5]:
# we need to do this to get all of the tickers for the portfolio and benchmark in one spot
portfolio_tickers = [
    ticker
    for portfolio in portfolios.values()
    for ticker in portfolio.keys()
]

benchmark_tickers = list(benchmarks.values())

benchmark_portfolio_tickers = [
    ticker
    for portfolio in benchmark_portfolios.values()
    for ticker in portfolio.keys()
]

all_tickers = sorted(set(portfolio_tickers + benchmark_tickers + benchmark_portfolio_tickers))

all_tickers

['AAPL',
 'AMZN',
 'AVGO',
 'BND',
 'CEG',
 'COP',
 'COST',
 'CVX',
 'D',
 'DUK',
 'EOG',
 'GIS',
 'GOOG',
 'KMB',
 'KO',
 'META',
 'MSFT',
 'NEE',
 'NSRGY',
 'NVDA',
 'PEP',
 'PG',
 'QQQ',
 'SLB',
 'SO',
 'SPY',
 'TBIL',
 'TGT',
 'TSLA',
 'UL',
 'VNQ',
 'WMT',
 'XOM',
 '^VIX']

## Historical Price Data

In [6]:
# Date range for the backtest
START_DATE = "2007-01-01"
END_DATE = "2026-05-31"

# Download price data from Yahoo Finance
price_data = yf.download(
    all_tickers,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

# Keep only closing prices
prices = price_data["Close"]

# Preview the data
prices.head()


1 Failed download:
['XOM']: OperationalError('database is locked')


Ticker,AAPL,AMZN,AVGO,BND,CEG,COP,COST,CVX,D,DUK,EOG,GIS,GOOG,KMB,KO,META,MSFT,NEE,NSRGY,NVDA,PEP,PG,QQQ,SLB,SO,SPY,TBIL,TGT,TSLA,UL,VNQ,WMT,XOM,^VIX
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2007-01-03,2.5086,1.9350,NaN,NaN,NaN,26.8307,36.1509,33.3582,18.6449,24.0406,20.7947,14.9543,11.5443,32.7018,13.4175,NaN,21.0284,7.7648,19.9298,0.5507,35.2565,37.0780,37.0389,39.6224,15.6363,98.8695,NaN,34.7060,NaN,16.1765,34.2982,10.6840,NaN,12.0400
2007-01-04,2.5643,1.9450,NaN,NaN,NaN,25.9966,36.9924,33.0339,18.5981,24.0909,20.8084,14.9438,11.9312,32.8741,13.4230,NaN,20.9932,7.7591,19.9072,0.5481,35.4982,36.7964,37.7413,38.6321,15.6448,99.0793,NaN,34.9549,NaN,16.0900,34.3426,10.7357,NaN,11.5100
2007-01-05,2.5460,1.9185,NaN,NaN,NaN,26.5277,36.5409,33.1608,18.3306,23.5382,21.4872,14.8345,12.0282,32.7449,13.3291,NaN,20.8735,7.6563,19.6133,0.5137,35.3858,36.4805,37.5614,38.5670,15.4080,98.2890,NaN,34.7607,NaN,15.8364,33.7067,10.6481,NaN,12.1400
2007-01-08,2.5586,1.8750,NaN,NaN,NaN,26.8740,36.6914,33.5838,18.2481,23.5759,21.3397,14.9996,11.9391,32.9315,13.4147,NaN,21.0777,7.6464,19.5681,0.5175,35.4645,36.5609,37.5871,38.2217,15.4503,98.7436,NaN,34.8092,NaN,15.7268,33.7556,10.5604,NaN,12.0000
2007-01-09,2.7711,1.8890,NaN,NaN,NaN,26.1697,36.9651,33.1984,18.2169,23.4629,21.0655,14.8240,11.9865,33.0128,13.4258,NaN,21.0988,7.6534,19.5455,0.5075,35.6107,36.4690,37.7756,37.9090,15.4926,98.6597,NaN,35.3615,NaN,15.6749,34.1826,10.6481,NaN,11.9100


## EDA Checks

Here we are going to check the dataset to see if everything is passed in correctly.

In [7]:
# Check the shape of the dataset
print("Rows and columns:", prices.shape)

# Show the first few rows
display(prices.head())

# Show the last few rows
display(prices.tail())

Rows and columns: (4883, 34)


Ticker,AAPL,AMZN,AVGO,BND,CEG,COP,COST,CVX,D,DUK,EOG,GIS,GOOG,KMB,KO,META,MSFT,NEE,NSRGY,NVDA,PEP,PG,QQQ,SLB,SO,SPY,TBIL,TGT,TSLA,UL,VNQ,WMT,XOM,^VIX
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2007-01-03,2.5086,1.9350,NaN,NaN,NaN,26.8307,36.1509,33.3582,18.6449,24.0406,20.7947,14.9543,11.5443,32.7018,13.4175,NaN,21.0284,7.7648,19.9298,0.5507,35.2565,37.0780,37.0389,39.6224,15.6363,98.8695,NaN,34.7060,NaN,16.1765,34.2982,10.6840,NaN,12.0400
2007-01-04,2.5643,1.9450,NaN,NaN,NaN,25.9966,36.9924,33.0339,18.5981,24.0909,20.8084,14.9438,11.9312,32.8741,13.4230,NaN,20.9932,7.7591,19.9072,0.5481,35.4982,36.7964,37.7413,38.6321,15.6448,99.0793,NaN,34.9549,NaN,16.0900,34.3426,10.7357,NaN,11.5100
2007-01-05,2.5460,1.9185,NaN,NaN,NaN,26.5277,36.5409,33.1608,18.3306,23.5382,21.4872,14.8345,12.0282,32.7449,13.3291,NaN,20.8735,7.6563,19.6133,0.5137,35.3858,36.4805,37.5614,38.5670,15.4080,98.2890,NaN,34.7607,NaN,15.8364,33.7067,10.6481,NaN,12.1400
2007-01-08,2.5586,1.8750,NaN,NaN,NaN,26.8740,36.6914,33.5838,18.2481,23.5759,21.3397,14.9996,11.9391,32.9315,13.4147,NaN,21.0777,7.6464,19.5681,0.5175,35.4645,36.5609,37.5871,38.2217,15.4503,98.7436,NaN,34.8092,NaN,15.7268,33.7556,10.5604,NaN,12.0000
2007-01-09,2.7711,1.8890,NaN,NaN,NaN,26.1697,36.9651,33.1984,18.2169,23.4629,21.0655,14.8240,11.9865,33.0128,13.4258,NaN,21.0988,7.6534,19.5455,0.5075,35.6107,36.4690,37.7756,37.9090,15.4926,98.6597,NaN,35.3615,NaN,15.6749,34.1826,10.6481,NaN,11.9100


Ticker,AAPL,AMZN,AVGO,BND,CEG,COP,COST,CVX,D,DUK,EOG,GIS,GOOG,KMB,KO,META,MSFT,NEE,NSRGY,NVDA,PEP,PG,QQQ,SLB,SO,SPY,TBIL,TGT,TSLA,UL,VNQ,WMT,XOM,^VIX
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.5900
2026-05-26,308.3300,265.2900,421.3432,72.9638,301.5700,116.5700,1002.9300,184.7100,66.6130,124.9700,136.2000,33.1600,384.6085,97.4317,79.9439,611.7730,416.0300,87.0127,100.9300,214.6099,144.1632,142.9600,729.4784,57.6776,94.0900,748.6613,49.8330,125.4300,433.5900,57.0700,96.3706,118.5700,NaN,17.0100
2026-05-27,310.8500,271.8500,421.1934,73.0336,288.6800,115.1300,1003.6900,182.4000,66.5338,125.3500,134.3000,33.6500,384.5985,98.8226,81.0964,634.6718,412.6700,87.0127,102.6100,212.3525,146.2018,147.4900,728.6493,56.2053,93.7400,748.5316,49.8430,128.3300,440.3600,58.0000,96.0732,118.5400,NaN,16.2900
2026-05-28,312.5100,274.0000,425.9059,73.1731,286.3100,114.9900,995.2000,183.0300,66.7120,123.7600,134.5800,33.8900,385.8878,98.7832,79.8942,634.7017,426.9900,86.6156,101.9400,214.0006,144.7669,145.9100,734.7925,54.8325,92.5200,752.6609,49.8500,128.6500,442.1000,57.0300,95.7065,118.9000,NaN,15.7400
2026-05-29,312.0600,270.6400,446.0640,73.2130,287.7500,113.9800,956.3200,182.4600,66.9400,122.7300,133.3800,33.8100,376.2036,96.2776,78.5032,631.9243,450.2400,86.3773,101.4400,210.8942,142.6888,143.5600,737.4995,54.2655,92.0500,754.5361,49.8600,127.0700,435.7900,56.4500,94.8639,115.7500,NaN,15.3200


Now we need to check which assets have missing values or limited history!

## Data Validation

The main reason we are checking this is to see wether each asset has enough historical price data for the backtest. Assets with later starting dates may limit which crisis periods can be analyzed fairly.

This is an important part because the project compares portfolio performance during historical finacial crisises, so each portfolio needs enough data to support a meaningful comparision.

In [8]:
missing_summary = pd.DataFrame({
    "Missing Count": prices.isna().sum(),
    "Missing Percent": prices.isna().mean() * 100,
    "First Valid Date": prices.apply(lambda col: col.first_valid_index()),
    "Last Valid Date": prices.apply(lambda col: col.last_valid_index())
})

missing_summary = missing_summary.sort_values("First Valid Date")

missing_summary

,Missing Count,Missing Percent,First Valid Date,Last Valid Date
Ticker,,,,
AAPL,1,0.0205,2007-01-03,2026-05-29
VNQ,1,0.0205,2007-01-03,2026-05-29
UL,1,0.0205,2007-01-03,2026-05-29
TGT,1,0.0205,2007-01-03,2026-05-29
SPY,1,0.0205,2007-01-03,2026-05-29
SO,1,0.0205,2007-01-03,2026-05-29
SLB,1,0.0205,2007-01-03,2026-05-29
QQQ,1,0.0205,2007-01-03,2026-05-29
PG,1,0.0205,2007-01-03,2026-05-29


## Calculate Daily Asset Returns

In [9]:
asset_returns = prices.pct_change(fill_method=None)

# Preview daily returns
asset_returns.head()

Ticker,AAPL,AMZN,AVGO,BND,CEG,COP,COST,CVX,D,DUK,EOG,GIS,GOOG,KMB,KO,META,MSFT,NEE,NSRGY,NVDA,PEP,PG,QQQ,SLB,SO,SPY,TBIL,TGT,TSLA,UL,VNQ,WMT,XOM,^VIX
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2007-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-01-04,0.0222,0.0052,NaN,NaN,NaN,-0.0311,0.0233,-0.0097,-0.0025,0.0021,0.0007,-0.0007,0.0335,0.0053,0.0004,NaN,-0.0017,-0.0007,-0.0011,-0.0047,0.0069,-0.0076,0.0190,-0.0250,0.0005,0.0021,NaN,0.0072,NaN,-0.0053,0.0013,0.0048,NaN,-0.0440
2007-01-05,-0.0071,-0.0136,NaN,NaN,NaN,0.0204,-0.0122,0.0038,-0.0144,-0.0229,0.0326,-0.0073,0.0081,-0.0039,-0.0070,NaN,-0.0057,-0.0133,-0.0148,-0.0627,-0.0032,-0.0086,-0.0048,-0.0017,-0.0151,-0.0080,NaN,-0.0056,NaN,-0.0158,-0.0185,-0.0082,NaN,0.0547
2007-01-08,0.0049,-0.0227,NaN,NaN,NaN,0.0131,0.0041,0.0128,-0.0045,0.0016,-0.0069,0.0111,-0.0074,0.0057,0.0064,NaN,0.0098,-0.0013,-0.0023,0.0074,0.0022,0.0022,0.0007,-0.0090,0.0027,0.0046,NaN,0.0014,NaN,-0.0069,0.0015,-0.0082,NaN,-0.0115
2007-01-09,0.0831,0.0075,NaN,NaN,NaN,-0.0262,0.0075,-0.0115,-0.0017,-0.0048,-0.0129,-0.0117,0.0040,0.0025,0.0008,NaN,0.0010,0.0009,-0.0012,-0.0195,0.0041,-0.0025,0.0050,-0.0082,0.0027,-0.0008,NaN,0.0159,NaN,-0.0033,0.0126,0.0083,NaN,-0.0075


## Build Weighted Portfolio Returns

We are going to use each portfolio's weights to create a single return series.

In [10]:
def build_weighted_portfolio_returns(returns_df, portfolio_dict):
    portfolio_returns = pd.DataFrame(index=returns_df.index)

    for portfolio_name, weights in portfolio_dict.items():
        tickers = list(weights.keys())
        weight_values = np.array(list(weights.values()))

        # Keep only dates where all assets in the portfolio have return data
        available_returns = returns_df[tickers].dropna()

        # Calculate weighted daily portfolio return
        weighted_returns = (available_returns * weight_values).sum(axis=1)

        # Store results
        portfolio_returns.loc[available_returns.index, portfolio_name] = weighted_returns

    return portfolio_returns

# Create daily returns for stakeholder portfolios
student_portfolio_returns = build_weighted_portfolio_returns(asset_returns, portfolios)

# Create daily returns for benchmark portfolio, like 60/40
benchmark_portfolio_returns = build_weighted_portfolio_returns(asset_returns, benchmark_portfolios)

# Create daily returns for individual benchmarks
individual_benchmark_returns = pd.DataFrame(index=asset_returns.index)

for benchmark_name, ticker in benchmarks.items():
    individual_benchmark_returns[benchmark_name] = asset_returns[ticker]

# Combine everything into one dataframe
combined_returns = pd.concat(
    [
        student_portfolio_returns,
        benchmark_portfolio_returns,
        individual_benchmark_returns
    ],
    axis=1
)

combined_returns = combined_returns.dropna(how="all")

combined_returns.head()

,Barbell Portfolio,Mega-Cap Technology Portfolio,Energy Portfolio,Real Estate Portfolio,Consumer Staples Portfolio,60/40 Portfolio,S&P 500,Nasdaq 100,Bonds
Date,,,,,,,,,
2007-01-04,NaN,NaN,NaN,0.0013,0.0033,NaN,0.0021,0.0190,NaN
2007-01-05,NaN,NaN,NaN,-0.0185,-0.0086,NaN,-0.0080,-0.0048,NaN
2007-01-08,NaN,NaN,NaN,0.0015,0.0016,NaN,0.0046,0.0007,NaN
2007-01-09,NaN,NaN,NaN,0.0126,0.0020,NaN,-0.0008,0.0050,NaN
2007-01-10,NaN,NaN,NaN,0.0129,0.0034,NaN,0.0033,0.0118,NaN


Now some quick tests to see if it shows!

In [11]:
print("Portfolio return data shape:", combined_returns.shape)

display(combined_returns.head())
display(combined_returns.tail())

combined_returns.describe()

Portfolio return data shape: (4880, 9)


,Barbell Portfolio,Mega-Cap Technology Portfolio,Energy Portfolio,Real Estate Portfolio,Consumer Staples Portfolio,60/40 Portfolio,S&P 500,Nasdaq 100,Bonds
Date,,,,,,,,,
2007-01-04,NaN,NaN,NaN,0.0013,0.0033,NaN,0.0021,0.0190,NaN
2007-01-05,NaN,NaN,NaN,-0.0185,-0.0086,NaN,-0.0080,-0.0048,NaN
2007-01-08,NaN,NaN,NaN,0.0015,0.0016,NaN,0.0046,0.0007,NaN
2007-01-09,NaN,NaN,NaN,0.0126,0.0020,NaN,-0.0008,0.0050,NaN
2007-01-10,NaN,NaN,NaN,0.0129,0.0034,NaN,0.0033,0.0118,NaN


,Barbell Portfolio,Mega-Cap Technology Portfolio,Energy Portfolio,Real Estate Portfolio,Consumer Staples Portfolio,60/40 Portfolio,S&P 500,Nasdaq 100,Bonds
Date,,,,,,,,,
2026-05-21,-0.0018,-0.0005,NaN,0.0019,-0.0046,0.0016,0.0020,0.0019,0.0011
2026-05-22,0.0002,-0.0004,NaN,0.0010,-0.0009,0.0027,0.0039,0.0042,0.0010
2026-05-27,-0.0019,0.0084,NaN,-0.0031,0.0146,0.0003,-0.0002,-0.0011,0.0010
2026-05-28,-0.0016,0.0093,NaN,-0.0038,-0.0055,0.0041,0.0055,0.0084,0.0019
2026-05-29,-0.0011,0.0037,NaN,-0.0088,-0.0169,0.0017,0.0025,0.0037,0.0005


,Barbell Portfolio,Mega-Cap Technology Portfolio,Energy Portfolio,Real Estate Portfolio,Consumer Staples Portfolio,60/40 Portfolio,S&P 500,Nasdaq 100,Bonds
count,953.0000,3525.0000,0.0000,4880.0000,4880.0000,4814.0000,4880.0000,4880.0000,4814.0000
mean,0.0003,0.0014,NaN,0.0004,0.0004,0.0003,0.0005,0.0007,0.0001
std,0.0039,0.0167,NaN,0.0186,0.0093,0.0076,0.0124,0.0140,0.0033
min,-0.0175,-0.1441,NaN,-0.1951,-0.0865,-0.0792,-0.1094,-0.1198,-0.0544
25%,-0.0018,-0.0067,NaN,-0.0066,-0.0040,-0.0025,-0.0041,-0.0052,-0.0014
50%,-0.0001,0.0021,NaN,0.0007,0.0006,0.0005,0.0007,0.0012,0.0002
75%,0.0018,0.0102,NaN,0.0076,0.0051,0.0037,0.0060,0.0076,0.0018
max,0.0372,0.1527,NaN,0.1701,0.0978,0.1038,0.1452,0.1216,0.0422


In [12]:
# M2 Data Summary Counts
# Creating row counts, date ranges, portfolio counts, and ticker counts for the M2 Data Summary.

summary_counts = {}

# Portfolio construction table
if "portfolio_reference" in globals():
    summary_counts["portfolio_reference_rows"] = len(portfolio_reference)
    summary_counts["unique_portfolios"] = portfolio_reference["Portfolio"].nunique() if "Portfolio" in portfolio_reference.columns else portfolio_reference["portfolio"].nunique()
    summary_counts["unique_tickers"] = portfolio_reference["Ticker"].nunique() if "Ticker" in portfolio_reference.columns else portfolio_reference["ticker"].nunique()

# Raw or adjusted prices
if "prices" in globals():
    summary_counts["prices_shape"] = prices.shape
    summary_counts["prices_rows_wide"] = prices.shape[0]
    summary_counts["prices_columns_wide"] = prices.shape[1]
    summary_counts["price_start_date"] = prices.index.min()
    summary_counts["price_end_date"] = prices.index.max()

if "prices_tidy" in globals():
    summary_counts["prices_tidy_rows"] = len(prices_tidy)
    summary_counts["prices_tidy_start_date"] = prices_tidy["date"].min()
    summary_counts["prices_tidy_end_date"] = prices_tidy["date"].max()

# Asset returns
if "asset_returns" in globals():
    summary_counts["asset_returns_shape"] = asset_returns.shape
    summary_counts["asset_returns_rows_wide"] = asset_returns.shape[0]
    summary_counts["asset_returns_columns_wide"] = asset_returns.shape[1]

if "asset_returns_tidy" in globals():
    summary_counts["asset_returns_tidy_rows"] = len(asset_returns_tidy)
    summary_counts["asset_returns_tidy_start_date"] = asset_returns_tidy["date"].min()
    summary_counts["asset_returns_tidy_end_date"] = asset_returns_tidy["date"].max()

# Portfolio returns
if "combined_returns" in globals():
    summary_counts["combined_returns_shape"] = combined_returns.shape
    summary_counts["combined_returns_rows"] = combined_returns.shape[0]
    summary_counts["combined_returns_columns"] = combined_returns.shape[1]
    summary_counts["combined_returns_start_date"] = combined_returns.index.min()
    summary_counts["combined_returns_end_date"] = combined_returns.index.max()

if "portfolio_returns_tidy" in globals():
    summary_counts["portfolio_returns_tidy_rows"] = len(portfolio_returns_tidy)
    summary_counts["portfolio_returns_tidy_start_date"] = portfolio_returns_tidy["date"].min()
    summary_counts["portfolio_returns_tidy_end_date"] = portfolio_returns_tidy["date"].max()

# Performance outputs
if "performance_summary" in globals():
    summary_counts["performance_summary_rows"] = len(performance_summary)

if "crisis_summary" in globals():
    summary_counts["crisis_summary_rows"] = len(crisis_summary)

# Print clean summary
for key, value in summary_counts.items():
    print(f"{key}: {value}")

portfolio_reference_rows: 31
unique_portfolios: 5
unique_tickers: 31
prices_shape: (4883, 34)
prices_rows_wide: 4883
prices_columns_wide: 34
price_start_date: 2007-01-03 00:00:00
price_end_date: 2026-05-29 00:00:00
asset_returns_shape: (4883, 34)
asset_returns_rows_wide: 4883
asset_returns_columns_wide: 34
combined_returns_shape: (4880, 9)
combined_returns_rows: 4880
combined_returns_columns: 9
combined_returns_start_date: 2007-01-04 00:00:00
combined_returns_end_date: 2026-05-29 00:00:00
